# Inference & Evaluation

> bioMONAI functions for model inference and evaluation


In [ ]:
#| default_exp inference

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export

# =================================
# Scientific / data
# =================================
import numpy as np
import pandas as pd

# =================================
# Visualization
# =================================
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# =================================
# Imaging
# =================================
# from skimage import util

# =================================
# PyTorch
# =================================
import torch.optim as toptim
from torch.cuda import is_available as is_cuda_available
from torch.nn.init import kaiming_normal_

# =================================
# fastai
# =================================
# import fastai.losses
# import fastai.metrics
# import fastai.optimizer

from fastai.callback.core import Callback
from fastai.callback.all import *

from fastai.data.all import (
    DataLoaders, Path, trainable_params, delegates,
    hasattrs, List, L, Normalize
)

from fastai.optimizer import Adam, OptimWrapper, Optimizer

from fastai.vision.all import (
    Any, BypassNewMeta, CSVLogger, ClassificationInterpretation,
    DataBlock, DisplayedTransform, Learner, ShowGraphCallback,
    create_vision_model, create_timm_model, default_split,
    get_c, ifnone, minimum, model_meta, slide, steep, store_attr, valley
)

# =================================
# fastcore
# =================================
from fastcore.script import risinstance

# =================================
# bioMONAI
# =================================
from bioMONAI.utils import *
from bioMONAI.datasets import download_medmnist

## Evaluation

The evaluation module provides functionalities for model evaluation, with several customizations available.

In [ ]:
#| export
def compute_losses(predictions, targets, loss_fn):
    """
    Compute the loss for each prediction-target pair.
    """
    return [loss_fn(p.unsqueeze(0), t.unsqueeze(0)).item() for p, t in zip(predictions, targets)]


def compute_metric(predictions, targets, metric_fn):
    """
    Compute the metric for each prediction-target pair.
    Handles cases where metric_fn has or does not have a 'func' attribute.
    """
    # Get the actual function to call (either metric_fn.func or metric_fn itself)
    metric_func = getattr(metric_fn, 'func', metric_fn)
    
    return [metric_func(p.unsqueeze(0), t.unsqueeze(0)).item() for p, t in zip(predictions, targets)]


def calculate_statistics(data):
    """
    Calculate key statistics for the data.
    """
    return {
        "Mean": np.mean(data),
        "Median": np.median(data),
        "Standard Deviation": np.std(data),
        "Min": np.min(data),
        "Max": np.max(data),
        "Q1": np.percentile(data, 25),
        "Q3": np.percentile(data, 75),
    }

In [ ]:
from numpy.random import standard_normal

In [ ]:
a = standard_normal(1000)

calculate_statistics(a)

{'Mean': np.float64(-0.01687829565636342),
 'Median': np.float64(0.0072560237539078185),
 'Standard Deviation': np.float64(1.0555940195555695),
 'Min': np.float64(-2.850353015330463),
 'Max': np.float64(3.9646292618110106),
 'Q1': np.float64(-0.7232186840855153),
 'Q3': np.float64(0.6719983225269198)}

In [ ]:
#| export
# Retrieve the 'coolwarm' colormap
coolwarm = plt.get_cmap('coolwarm')
# Create a new colormap using only the warm colors
warm_cmap = LinearSegmentedColormap.from_list('warm_coolwarm', coolwarm(np.linspace(0.5, 1, coolwarm.N // 2)))

Evaluate_model and evaluate_classification_model are two classes created in order to integrate the evaluation process in a single computation. Evaluate_model can be used on any type of task, whereas evaluate_classification_model is specifically designed for classification tasks. 

In [ ]:
#| export 
def evaluate_model(trainer:Learner,                                 # The model trainer object with a get_preds method.
                   test_data:DataLoaders=None,              # DataLoader containing test data.
                   loss=None,                               # Loss function to evaluate prediction-target pairs.
                   metrics=None,                            # Single metric or a list of metrics to evaluate. 
                   bw_method=0.3,                           # Bandwidth method for KDE. 
                   show_graph=True,                         # Boolean flag to show the histogram and KDE plot.
                   show_table=True,                         # Boolean flag to show the statistics table.
                   show_results=True,                       # Boolean flag to show model results on test data. 
                   as_dataframe=True,                       # Boolean flag to display table as a DataFrame. 
                   cmap='magma',                            # Colormap for visualization.
                   use_plotly=True,                         # Boolean flag to use Plotly for interactive plots instead of Matplotlib.
                   ):
    """
    Calculate and optionally plot the distribution of loss values from predictions
    made by the trainer on test data, with an optional table of key statistics.
    """
    from bioMONAI.visualize import plot_histogram_and_kde, display_statistics_table, plot_dist

    out = dict()
    
    if loss is None:
        loss = trainer.loss_func
        
    if test_data is None:
        p, t = trainer.get_preds()
        # Show results for test data
        if show_results:
            trainer.show_results(cmap=cmap)
    else:
        p, t = trainer.get_preds(dl=test_data)
        # Show results for test data
        if show_results:
            trainer.show_results(dl=test_data, cmap=cmap)

    # Calculate loss for each prediction-target pair
    losses = compute_losses(p, t, loss)
    loss_stats = calculate_statistics(losses)
    loss_name = loss.__class__.__name__  # Get loss function name
    out[loss_name] = losses    
    if show_graph:
        if use_plotly:
            plot_dist(losses, title=loss_name)
        else:
            plot_histogram_and_kde(losses, loss_stats, bw_method, loss_name)

    if show_table:
        display_statistics_table(loss_stats, loss_name, as_dataframe=as_dataframe)
            
    if metrics is not None:
            if not isinstance(metrics, list):
                metrics = [metrics]
            # Loop through each metric
            for metric in metrics:
                # Calculate metric values for each prediction-target pair
                metric_values = compute_metric(p, t, metric)
                metric_stats = calculate_statistics(metric_values)         
                # Get the name of the metric function
                metric_name = getattr(metric, 'func', metric).__name__  # Support AvgMetric or regular functions                
                out[metric_name] = metric_values       
                if show_graph:
                    if use_plotly:
                        plot_dist(losses, title=loss_name)
                    else:
                        plot_histogram_and_kde(losses, loss_stats, bw_method, loss_name)
                if show_table:
                    display_statistics_table(metric_stats, metric_name, as_dataframe=as_dataframe)

    return out


In [ ]:
#| export
def evaluate_classification_model(trainer:Learner,              # The trained model (learner) to evaluate.
                                  test_data:DataLoaders=None,   # DataLoader with test data for evaluation. If None, the validation dataset is used.
                                  loss_fn=None,                 # Loss function used in the model for ClassificationInterpretation. If None, the loss function is loaded from trainer.
                                  most_confused_n:int=1,        # Number of most confused class pairs to display. 
                                  normalize:bool=True,          # Whether to normalize the confusion matrix.
                                  act=None,                     # Apply activation to predictions, defaults to `self.loss_func`'s activation
                                  metrics=None,                 # Single metric or a list of metrics to evaluate. 
                                  bw_method=0.3,                # Bandwidth method for KDE. 
                                  show_graph=True,              # Boolean flag to show the histogram and KDE plot.
                                  show_table=True,              # Boolean flag to show the statistics table.
                                  show_results=True,            # Boolean flag to show model results on test data. 
                                  as_dataframe=True,            # Boolean flag to display table as a DataFrame. 
                                  cmap=warm_cmap,               # Color map for the confusion matrix plot. 
                                  use_plotly=True,              # Boolean flag to use Plotly for interactive plots instead of Matplotlib.
                                  ):
    """
    Evaluates a classification model by displaying results, confusion matrix, and most confused classes.
    """
    from bioMONAI.visualize import plot_histogram_and_kde, display_statistics_table, plot_dist

    out = dict()
    
    if loss_fn is None:
            loss_fn = trainer.loss_func
    
    # Interpret the results on test data
    if test_data is None:
        class_int = ClassificationInterpretation.from_learner(trainer, act=act)
        p, t = trainer.get_preds(act=act)
        # Show results for test data
        if show_results:
            trainer.show_results()
    else:
        class_int = ClassificationInterpretation(trainer, test_data, loss_fn, act=act)
        p, t = trainer.get_preds(dl=test_data, act=act)
        # Show results for test data
        if show_results:
            trainer.show_results(dl=test_data)
    
    # Plot the confusion matrix
    class_int.plot_confusion_matrix(normalize=normalize, cmap=cmap)
    
    # Print Classification report
    class_int.print_classification_report()
    
    # Show the most confused classes
    out['most_confused'] = pd.DataFrame(class_int.most_confused(most_confused_n), 
                                        columns=["Actual Class", "Predicted Class", "Count"])
    print("\nMost Confused Classes:")
    display(out['most_confused'])

    # Calculate loss for each prediction-target pair
    losses = compute_losses(p, t, loss_fn)
    loss_stats = calculate_statistics(losses)
    loss_name = loss_fn.__class__.__name__  # Get loss function name
    out[loss_name] = losses
    if show_graph:
            if use_plotly:
                plot_dist(losses, title=loss_name)
            else:
                plot_histogram_and_kde(losses, loss_stats, bw_method, loss_name)
    if show_table:
        display_statistics_table(loss_stats, loss_name, as_dataframe=as_dataframe)
            
    if metrics is not None:
            if not isinstance(metrics, list):
                metrics = [metrics]
            # Loop through each metric
            for metric in metrics:
                # Calculate metric values for each prediction-target pair
                metric_values = compute_metric(p, t, metric)
                metric_stats = calculate_statistics(metric_values)                
                # Get the name of the metric function
                metric_name = getattr(metric, 'func', metric).__name__  # Support AvgMetric or regular functions 
                out[metric_name] = metric_values                      
                if show_graph:
                    if use_plotly:
                        plot_dist(losses, title=loss_name)
                    else:
                        plot_histogram_and_kde(losses, loss_stats, bw_method, loss_name)
                if show_table:
                    display_statistics_table(metric_stats, metric_name, as_dataframe=as_dataframe)
    
    return out


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()